In [0]:
gl_df = spark.table("pl_workforce_analysis.silver.general_ledgers")
accounts_df = spark.table("pl_workforce_analysis.silver.accounts")
company_df = spark.table("pl_workforce_analysis.silver.company")
payroll_df = spark.table("pl_workforce_analysis.silver.payroll")
employee_df = spark.table("pl_workforce_analysis.silver.employees")
department_df = spark.table("pl_workforce_analysis.silver.departments")

In [0]:
from pyspark.sql.functions import col

financial_df = gl_df.join(accounts_df, "account_id", "left").join(company_df, "company_id", "left")
financial_df = financial_df.withColumn("final_amount",col("amount") * col("pnl_sign"))

display(financial_df)

In [0]:
from pyspark.sql.functions import year, month

financial_df = financial_df.withColumn("year", year("entry_date")).withColumn("month", month("entry_date"))
display(financial_df)

In [0]:
from pyspark.sql.functions import col, sum as _sum, round

revenue_df = financial_df.filter(col("pnl_section") == "Revenue").groupBy("year", "month").agg(round(_sum("final_amount"), 2).alias("revenue")).orderBy("year", "month")

display(revenue_df)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, col, round

window_spec = Window.orderBy("year", "month")

revenue_df = revenue_df.withColumn("prev_revenue", lag("revenue").over(window_spec)).withColumn("growth_pct", round(((col("revenue") - col("prev_revenue")) / col("prev_revenue")) * 100, 2))

display(revenue_df)

In [0]:
from pyspark.sql.functions import col, sum as _sum, round

cogs_df = financial_df.filter(col("pnl_section") == "COGS").groupBy("year", "month").agg(round(_sum("final_amount"), 2).alias("cogs")).orderBy("year", "month").withColumn("cogs", col("cogs") * -1)

display(cogs_df)

In [0]:
cogs_total_df = financial_df.filter(col("pnl_section") == "COGS").groupBy("year", "month").agg(round(_sum("final_amount") * -1, 2).alias("cogs"))

gpm_df = revenue_df.join(cogs_total_df,["year", "month"],"left").withColumn("gross_profit_margin",round(((col("revenue") - col("cogs")) / col("revenue")) * 100, 2))

display(gpm_df)

In [0]:
opex_df = financial_df.filter(col("pnl_section") == "Operating Expense").groupBy("company_id", "account_id").agg(round(_sum("final_amount") * -1, 2).alias("expense")).orderBy("company_id","account_id")

display(opex_df)

In [0]:
from pyspark.sql.functions import avg, round

p = payroll_df.alias("p")
e = employee_df.alias("e")

workforce_df = p.join(e, "employee_id", "left")
avg_comp_df = workforce_df.groupBy("p.company_id", "position").agg(round(avg("total_payroll_cost"), 2).alias("avg_compensation"))

display(avg_comp_df)

In [0]:
from pyspark.sql.functions import sum as _sum, round

profit_df = financial_df.groupBy("year", "month").agg(round(_sum("final_amount"), 2).alias("net_profit")).orderBy("year", "month")

display(profit_df)

In [0]:
from pyspark.sql.functions import sum as _sum, round

overtime_bonus_df = workforce_df.groupBy("p.department_id").agg(
        round(_sum("overtime_pay"), 2).alias("total_overtime"),
        round(_sum("bonus"), 2).alias("total_bonus"),
        round(_sum("overtime_pay") / _sum("base_salary"), 2).alias("overtime_ratio")
    )

display(overtime_bonus_df)

In [0]:
from pyspark.sql.functions import sum as _sum, round

dept_cost_df = workforce_df.groupBy("p.department_id").agg(
        round(_sum("total_payroll_cost"), 2).alias("department_cost")
    )

display(dept_cost_df)

In [0]:
from pyspark.sql.functions import col

headcount_df = employee_df.filter(col("is_active") == True).groupBy("department_id").count().orderBy("department_id")

display(headcount_df)

In [0]:
from pyspark.sql import functions as F
import builtins

total_payroll = payroll_df.agg(F.sum("total_payroll_cost").alias("total_payroll")).collect()[0][0]
total_revenue = revenue_df.agg(F.sum("revenue").alias("total_revenue")).collect()[0][0]

payroll_ratio = (total_payroll / total_revenue) * 100

result = builtins.round(payroll_ratio, 2)
print(f"Payroll Ratio: {result}%")